# K-DAREK on 10·cos(x): extended vs. non-extended knot grid

`kan_extend` controls whether K-DAREK's internal KAN spline grid is padded with
extra boundary knots beyond the training-data range. This notebook trains the
same model with `kan_extend=True` and `kan_extend=False` on the same data and
compares the fits directly, then builds a K-DAREK instance with a specific,
fixed **200-parameter** budget.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from kdarek import KDAREK, Dataset, set_seed, count_parameters, check_error_violation

seed = 12
cos_dataset = Dataset(fx=lambda x: 10 * np.cos(x), n=50, fix=True, seed=seed)
x_test, y_test = cos_dataset['test_input'], cos_dataset['test_label']

plt.scatter(cos_dataset['train_input'], cos_dataset['train_label'], label='train', alpha=0.5)
plt.plot(x_test, y_test, '--', color='k', alpha=0.5, label='true')
plt.legend()

## Train with `kan_extend=True` and `kan_extend=False`

In [ ]:
results = {}
for extend in [True, False]:
    set_seed(seed)
    kdk = KDAREK([1, 5], [5, 1], kan_grid=8, kan_k=3, kan_base_fun='silu', kan_seed=seed,
                 device='cpu', L_l=1.0, symbolic_enabled=False, auto_save=False, kan_extend=extend)
    kdk.fit(cos_dataset, lr=0.1, steps=1000, lamb=0.0, nonfixknot=True, seed_knots=42,
            rand_method='Kmean', scheduler='dec', step_sch=50, gamma=0.9, verbose=False)

    yhat, u = kdk.predict(x_test, L_k=10, L_1=10)
    err, vio, umean = check_error_violation(x_test.numpy(), y_test.numpy(),
                                             yhat.detach().numpy(), u.detach().numpy())

    knots_x = kdk.samples['xi'].flatten().sort().values
    knots_y = kdk.predict(knots_x.unsqueeze(1))[0].detach()

    results[extend] = dict(kdk=kdk, yhat=yhat.detach(), u=u.detach(), err=float(err),
                            vio=float(vio), umean=float(umean), knots_x=knots_x, knots_y=knots_y)
    print(f"kan_extend={extend}: RMSE={err:.4f}  violation={vio*100:.2f}%  mean_u={umean:.4f}")

`kan_extend=False` combined with adaptive (`nonfixknot=True`, K-means) knot
placement produces a noticeably worse, boundary-degraded fit here — the grid
the KAN spline settles on doesn't sanely cover the training domain edges
without the extra padding knots (this was traced to an actually malformed
knot grid, not just a slightly worse fit, while building this release).

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, extend in zip(axs, [True, False]):
    r = results[extend]
    ax.plot(x_test, y_test, '--', color='k', label='True')
    ax.plot(x_test, r['yhat'], color='blue', label='K2DAREK pred.')
    lb, ub = r['yhat'] - r['u'], r['yhat'] + r['u']
    ax.fill_between(x_test.flatten(), lb.flatten(), ub.flatten(), color='blue', alpha=0.2, label='bounds')
    ax.scatter(r['knots_x'], r['knots_y'], color='blue', zorder=10, label='knots')
    ax.set_title(f"kan_extend={extend}  (RMSE={r['err']:.3f}, vio={r['vio']*100:.1f}%)")
    ax.legend()
plt.tight_layout()

## A fixed 200-parameter K-DAREK model

`mlp_width=[1,7], kan_width=[7,1], kan_grid=4, kan_k=3` was found (by a small
search over widths/grid sizes) to give this package's K-DAREK architecture
**exactly 200 trainable parameters** — the same budget used for the
matched-parameter-count baseline comparison elsewhere in this project. Note:
the exact width/grid combination that hits 200 params is architecture-specific
(a different, incompatible K-DAREK implementation variant used for that
comparison reaches 200 params at a different width/grid setting); `count_parameters`
below is what actually verifies the budget for whatever code you're running,
rather than trusting a hardcoded width/grid combination to still be right.

In [ ]:
set_seed(47)
kdk200 = KDAREK([1, 7], [7, 1], kan_grid=4, kan_k=3, kan_base_fun='identity', L_l=3,
                seed=47, device='cpu', symbolic_enabled=False, auto_save=False, kan_extend=True)

n_params = count_parameters(kdk200)
print(f"K-DAREK parameter count: {n_params}")
assert n_params == 200

kdk200.fit(cos_dataset, lr=0.1, steps=1000, lamb=0.0, nonfixknot=True, seed_knots=42,
           rand_method='Kmean', scheduler='dec', step_sch=50, gamma=0.9, verbose=False)

yhat200, u200 = kdk200.predict(x_test, L_k=10, L_1=10)
err200, vio200, umean200 = check_error_violation(x_test.numpy(), y_test.numpy(),
                                                  yhat200.detach().numpy(), u200.detach().numpy())
print(f"RMSE={err200:.4f}  violation={vio200*100:.2f}%  mean_u={umean200:.4f}")

In [ ]:
lb200, ub200 = (yhat200 - u200).detach(), (yhat200 + u200).detach()

plt.figure(figsize=(7, 5))
plt.plot(x_test, y_test, '--', color='k', label='True')
plt.plot(x_test, yhat200.detach(), color='blue', label='K2DAREK pred. (200 params)')
plt.fill_between(x_test.flatten(), lb200.flatten(), ub200.flatten(), color='blue', alpha=0.2, label='bounds')
# The Lipschitz bound widens sharply near the domain edges (few nearby
# knots to anchor it) -- clip the view to where the fit itself is legible;
# the raw band is still there in lb200/ub200 if you want the full extent.
plt.ylim(-20, 20)
plt.title(f"Fixed 200-parameter K-DAREK (RMSE={err200:.3f})")
plt.legend()